## LLM model
`model = ModelInterface(`
    `model_id,`
    `prams,`
    `credentials,`
    `project_id`
`)`

`model_id = 'meta-llama/llma-302-90b-vision-instruct'`
-- the model is an instruct model, since the name has '-insturct'

`parameters = {GenParams.MAS_NEW_TOKEN: 256,GenParams.TEMPREATURE: 0.2}`

`credentials = {"url": "https://s-south.ml.could.ibm.com"}`

`project_id = "skills-network"`

- to run the model 
    - `model.generate()`
    - `print(msg['result][0]['generated_text'])`


## Chat Model
- force the model to LangChain compatable
    - text-in, text-out model as expected by the LangChain.
    - `WhatsonxLLM(model)`
    - `print(llama_ll.invoke("who is man's best friend?"))`

## Chat messages
- `SystemMessage` 
    - have high weightage, it has presidence over the Human message
    - During RHLF models are thught that the system messages are Master rules
    - Set the presona and the bounderies
- `HumanMessage`
    - task description
- `AIMessage`
    - LLM's reponse to the HumanMessage
    - Can be used for "One-shot" or "Few-shot" learning

`from langchain_cre.messages import HuamMessage, SystemMessage, AIMessage`



## Prompt Templates
- input parameters to the model

### String prompt templates
- to format a single string
- for simple inputs

In [3]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Tell me one {adjective} joke about {topic}"
    )

input_ = {
    "adjective": "funny",
    "topic" : "cats"
}

prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

### Chat prompt templates
- designed to work with chat models
- can assign various roles to the messates
    - system
    - human
    - ai

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about{topic}")
])

input_ = {"topic" : "cates"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke aboutcates', additional_kwargs={}, response_metadata={})])


### MessagePlaceholder
- Special tool for ChatPromptTemplate
- dynamic container for a list of messages
- standard place holder like {topic} expect a single string
- MessagePlaceholder excepts an array of message objects like HumanMessage or AIMessage

#### purpose
- can inject entire history of messages into a template
- code below

|Feature|{variable_name} (String)|MessagesPlaceholder|
|---|---|---|
|Expected Data | A single string. | A list of Message objects.|
|Result | Replaces text inside a message. | Adds multiple messages to the list.|
|Best For,"Keywords | topics, names." | "Chat history, Agent scratches, memory."|

In [5]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    # This is where the magic happens:
    MessagesPlaceholder(variable_name="chat_history"), 
    ("human", "{input}"),
])

# When you invoke this, you pass a LIST of messages for "chat_history"
# and a STRING for "input".

ModuleNotFoundError: No module named 'langchain.prompts'

## Output parsers
- conver the output of the LLM to a more suitable form
    - like to CSV or Json

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

import os


gemini_api = os.getenv("GEMINI")

# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_tokens=500,
    google_api_key=gemini_api
)

In [11]:
# JSON Parser
from langchain_core.output_parsers import JsonOutputParser

# Need ot import a BaseModel and Field form langchain to generate the model output
from pydantic import BaseModel, Field

class Joke(BaseModel):
    setup: str = Field(description = "Question to setup a joke")
    punchline: str = Field(description = "Answer to the joke")


joke_query = "Tell me a joke"

output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    input_variables = ["joke_query"],
    template = "Generate a joke in JSON format with the following fields: {format_instructions}\n\n{joke_query}",
    partial_variables = {"format_instructions": format_instructions}
)

chain = prompt | llm | output_parser

chain.invoke({"joke_query": joke_query})


{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}

- need format instructions
- says LLMs to produce output in a pariticular way that is understandable to the output parser
- in PromptTemplate
    - partial_variables = {"format_insturctions": output_parser.get_fromat_instructions()}

In [ ]:
print(format_instructions)